# dfguard — Polars quickstart

This notebook demonstrates the full dfguard Polars API end-to-end.

> **Note:** `arm()` patches all annotated functions in a module at import time and does not work inside notebooks. Use `@dfg.enforce` explicitly per function — that is the correct approach for notebook usage.

**Requirements:** `pip install 'dfguard[polars]'`

In [1]:
import polars as pl

import dfguard.polars as dfg
from dfguard.polars import Optional

## 1. Schema definition

Two ways to declare a schema: class-based `PolarsSchema` and `schema_of()` from a live DataFrame.

Fields use **assignment form** (`=`) to avoid Pylance `reportInvalidTypeForm` warnings on Polars dtype instances.

Polars supports first-class nested types: `pl.List`, `pl.Struct`, `pl.Array`. dfguard enforces them all.

`Optional[dtype]` marks a field as intentionally nullable. dfguard checks dtype but NOT null presence or absence.

In [2]:
# Class-based schema declaration (assignment form).
# Nested type: pl.List(pl.Struct({...})) — Polars native nested type.
class OrderSchema(dfg.PolarsSchema):
    order_id   = pl.Int64
    customer   = pl.String
    amount     = pl.Float64
    active     = pl.Boolean
    # List of structs: each order has one or more line items.
    line_items = pl.List(pl.Struct({
        "sku":      pl.String,
        "quantity": pl.Int32,
        "price":    pl.Float64,
    }))
    zip_code   = Optional[pl.String]   # nullable: dtype enforced, nulls allowed

print("Schema struct:", OrderSchema.to_struct())

Schema struct: {'order_id': Int64, 'customer': String, 'amount': Float64, 'active': Boolean, 'line_items': List(Struct({'sku': String, 'quantity': Int32, 'price': Float64})), 'zip_code': String}


In [3]:
# Build a matching DataFrame
orders_df = pl.DataFrame({
    "order_id":   [101, 102],
    "customer":   ["Alice", "Bob"],
    "amount":     [24.48, 14.00],
    "active":     [True, False],
    "line_items": [
        [{"sku": "A1", "quantity": 2, "price": 9.99},
         {"sku": "B2", "quantity": 1, "price": 4.50}],
        [{"sku": "C3", "quantity": 3, "price": 14.00}],
    ],
    "zip_code":   ["94107", None],
}, schema={
    "order_id":   pl.Int64,
    "customer":   pl.String,
    "amount":     pl.Float64,
    "active":     pl.Boolean,
    "line_items": pl.List(pl.Struct({"sku": pl.String, "quantity": pl.Int32, "price": pl.Float64})),
    "zip_code":   pl.String,
})
orders_df

order_id,customer,amount,active,line_items,zip_code
i64,str,f64,bool,list[struct[3]],str
101,"""Alice""",24.48,true,"[{""A1"",2,9.99}, {""B2"",1,4.5}]","""94107"""
102,"""Bob""",14.0,false,"[{""C3"",3,14.0}]",null


In [4]:
# schema_of() captures an exact schema from a live DataFrame.
# The resulting type requires exactly the captured columns — no extras.
# It also works with pl.LazyFrame.
CapturedSchema = dfg.schema_of(orders_df)
print("isinstance check (exact match):", isinstance(orders_df, CapturedSchema))

# LazyFrame works too
lazy_schema = dfg.schema_of(orders_df.lazy())
print("LazyFrame isinstance check:", isinstance(orders_df, lazy_schema))

isinstance check (exact match): True
LazyFrame isinstance check: True


## 2. Schema inheritance

In [5]:
# EnrichedOrderSchema inherits all fields from OrderSchema and adds revenue.
class EnrichedOrderSchema(OrderSchema):
    revenue = pl.Float64

print("Enriched fields:", list(EnrichedOrderSchema.to_struct().keys()))
assert "order_id" in EnrichedOrderSchema.to_struct()
assert "revenue"  in EnrichedOrderSchema.to_struct()
print("Inheritance confirmed.")

Enriched fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code', 'revenue']
Inheritance confirmed.


## 3. `@dfg.enforce` decorator

Two modes:
- `@dfg.enforce` (default, `subset=True`): the DataFrame must have at least the declared columns; extras are fine.
- `@dfg.enforce(subset=False)`: exact match — the DataFrame must have only the declared columns.

In [6]:
# subset=True (default): extra columns are fine.
@dfg.enforce
def compute_revenue(df: OrderSchema) -> pl.DataFrame:
    return df.with_columns(revenue=pl.col("amount") * 1.1)

enriched_df = compute_revenue(orders_df)
print("compute_revenue passed, columns:", enriched_df.columns)

compute_revenue passed, columns: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code', 'revenue']


In [7]:
# subset=False: exact match — only the declared columns allowed.
@dfg.enforce(subset=False)
def write_final(df: OrderSchema) -> None:
    print(f"write_final called with {len(df)} rows, columns: {df.columns}")

# PASSING: orders_df has exactly the OrderSchema columns.
write_final(orders_df)

# FAILING: enriched_df has an extra 'revenue' column.
try:
    write_final(enriched_df)
except TypeError as e:
    print(f"\nCaught expected error:\n{e}")

write_final called with 2 rows, columns: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']

Caught expected error:
Schema mismatch in write_final() argument 'df':
  expected: OrderSchema
  received: order_id:Int64, customer:String, amount:Float64, active:Boolean, line_items:List(Struct({'sku': String, 'quantity': Int32, 'price': Float64})), zip_code:String, revenue:Float64


In [8]:
# FAILING: wrong dtype — amount as string instead of Float64.
bad_df = orders_df.with_columns(pl.col("amount").cast(pl.String))

try:
    compute_revenue(bad_df)
except TypeError as e:
    print(f"Caught expected error:\n{e}")

Caught expected error:
Schema mismatch in compute_revenue() argument 'df':
  expected: OrderSchema
  received: order_id:Int64, customer:String, amount:String, active:Boolean, line_items:List(Struct({'sku': String, 'quantity': Int32, 'price': Float64})), zip_code:String


## 4. `assert_valid` for load-time validation

In [9]:
# PASSING
OrderSchema.assert_valid(orders_df)
print("assert_valid passed.")

# PASSING with subset=True (default): extra columns are fine.
OrderSchema.assert_valid(enriched_df, subset=True)
print("assert_valid(subset=True) passed on DataFrame with extra column.")

# FAILING with subset=False: enriched_df has 'revenue' not in OrderSchema.
try:
    OrderSchema.assert_valid(enriched_df, subset=False)
except dfg.SchemaValidationError as e:
    print(f"\nCaught expected error:\n{e}")

assert_valid passed.
assert_valid(subset=True) passed on DataFrame with extra column.

Caught expected error:
Schema validation failed:
  ✗ Unexpected column 'revenue' (strict mode)

Schema evolution (most recent last):
  [input]  {order_id:Int64, customer:String, amount:Float64, active:Boolean, line_items:List(Struct({'sku': String, 'quantity': Int32, 'price': Float64})), zip_code:String, revenue:Float64}  (no schema change)


## 5. Schema utilities: `from_struct`, `to_code`, `diff`, `empty`

In [10]:
# from_struct: generate a PolarsSchema from a Polars schema dict.
InferredSchema = dfg.PolarsSchema.from_struct(dict(orders_df.schema), name="InferredOrderSchema")
print("from_struct fields:", list(InferredSchema.to_struct().keys()))

from_struct fields: ['order_id', 'customer', 'amount', 'active', 'line_items', 'zip_code']


In [11]:
# to_code: generate Python source for the schema class.
print(InferredSchema.to_code())

import polars as pl
from dfguard.polars import PolarsSchema, Optional

class InferredOrderSchema(PolarsSchema):
    order_id: pl.Int64
    customer: pl.String
    amount: pl.Float64
    active: pl.Boolean
    line_items: pl.List(pl.Struct({"sku": pl.String, "quantity": pl.Int32, "price": pl.Float64}))
    zip_code: pl.String


In [12]:
# diff: show differences between two schemas.
print(OrderSchema.diff(EnrichedOrderSchema))
print()
print(OrderSchema.diff(OrderSchema))

Diff OrderSchema -> EnrichedOrderSchema:
  Missing column 'revenue' (expected Float64)

OrderSchema and OrderSchema are identical


In [13]:
# empty: zero-row DataFrame with the correct dtypes.
empty_df = OrderSchema.empty()
print("empty() shape:", empty_df.shape)
print("empty() schema:\n", empty_df.schema)

empty() shape: (0, 6)
empty() schema:
 Schema([('order_id', Int64), ('customer', String), ('amount', Float64), ('active', Boolean), ('line_items', List(Struct({'sku': String, 'quantity': Int32, 'price': Float64}))), ('zip_code', String)])


## 6. Schema history via `dfg.dataset()`

`dfg.dataset()` wraps a DataFrame in a tracking object. Each mutating operation records a schema change in an immutable history log.

In [14]:
ds = dfg.dataset(orders_df)

# Each step returns a new dataset instance; the original is unchanged.
ds2 = ds.with_columns(revenue=pl.col("amount") * 1.1)
ds3 = ds2.rename({"customer": "customer_name"})
ds4 = ds3.drop("zip_code")

# Print schema evolution.
ds4.schema_history.print()

────────────────────────────────────────────────────────────
Schema Evolution
────────────────────────────────────────────────────────────
  [ 0] input
       {order_id:Int64, customer:String, amount:Float64, active:Boolean, line_items:List(Struct({'sku': String, 'quantity': Int32, 'price': Float64})), zip_code:String}  (no schema change)
  [ 1] with_columns
       added: revenue
  [ 2] rename
       added: customer_name | dropped: customer
  [ 3] drop
       dropped: zip_code
────────────────────────────────────────────────────────────


## 7. Validate and check error messages (both passing and failing)

In [15]:
# validate() returns a list of errors without raising.
# PASSING: empty list.
errors = OrderSchema.validate(orders_df)
print("Errors on valid DataFrame:", errors)

# FAILING: missing column + wrong dtype.
broken_df = orders_df.drop("order_id").with_columns(pl.col("amount").cast(pl.String))
errors = OrderSchema.validate(broken_df)
print("\nErrors on broken DataFrame:")
for err in errors:
    print(" -", err)

Errors on valid DataFrame: []

Errors on broken DataFrame:
 - Missing column 'order_id' (expected Int64)
 - Column 'amount': type mismatch: expected Float64, got String


In [16]:
# FAILING: nested type mismatch — Int64 instead of Int32 in list struct.
wrong_schema_df = orders_df.with_columns(
    pl.col("line_items").cast(
        pl.List(pl.Struct({"sku": pl.String, "quantity": pl.Int64, "price": pl.Float64}))
    )
)
errors = OrderSchema.validate(wrong_schema_df)
print("Nested type errors:")
for err in errors:
    print(" -", err)

Nested type errors:
 - Column 'line_items': type mismatch: expected List(Struct({'sku': String, 'quantity': Int32, 'price': Float64})), got List(Struct({'sku': String, 'quantity': Int64, 'price': Float64}))


## 8. Putting it together: a typed pipeline

In [17]:
class ReportSchema(EnrichedOrderSchema):
    customer_name = pl.String

@dfg.enforce
def enrich(df: OrderSchema) -> pl.DataFrame:
    return df.with_columns(
        revenue=pl.col("amount") * 1.1,
        customer_name=pl.col("customer"),
    )

@dfg.enforce
def summarise(df: ReportSchema) -> pl.DataFrame:
    return df.select(["order_id", "customer_name", "revenue"])

result = summarise(enrich(orders_df))
print(result)

shape: (2, 3)
┌──────────┬───────────────┬─────────┐
│ order_id ┆ customer_name ┆ revenue │
│ ---      ┆ ---           ┆ ---     │
│ i64      ┆ str           ┆ f64     │
╞══════════╪═══════════════╪═════════╡
│ 101      ┆ Alice         ┆ 26.928  │
│ 102      ┆ Bob           ┆ 15.4    │
└──────────┴───────────────┴─────────┘
